[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/microscopy-processing/N2N-SPRVS/blob/main/N2N-SPRVS.ipynb)

# Generate similar_X (epfl_30nm)

In [ ]:
import numpy as np
import mrcfile
import cv2
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from collections import namedtuple

In [ ]:
%run _params_.ipynb

In [ ]:
REO_folder="/home/vruiz/REO/Experimento_saltoZ/epfl_30nm/"

In [ ]:
N = 1

In [ ]:
import os; cwd = os.getcwd(); print("Current Working Directory:", cwd)

In [ ]:
Args = namedtuple("args", ["X", "similar_X"])
args = Args("/home/vruiz/Tomograms/tomograms/Experimento_saltoZ/epfl_30nm.mrc", "001.mrc")

In [ ]:
X = mrcfile.open(args.X).data

In [ ]:
X.shape

In [ ]:
farneback_params = dict(
    pyr_scale=0.5,
    levels=3,
    winsize=WIN_SIZE,
    iterations=3,
    poly_n=POLY_N,
    poly_sigma=POLY_SIGMA,
    flags=0
)

In [ ]:
# https://stackoverflow.com/questions/62436299/how-to-lightly-shuffle-a-list-in-python
orderliness = 0.75

def tuplify(x, y):
  return (orderliness * y + np.random.normal(0, 1), x)

def shake(x, y, std_dev=1.0):
  displacements = np.random.normal(0, std_dev, len(x))
  #print(f"{np.min(displacements):.2f} {np.average(np.abs(displacements)):.2f} {np.max(displacements):.2f}", end=' ')
  return np.stack((y + displacements, x), axis=1)

def randomize(slice, mean=0.0, std_dev=1.0):
  #print(slice.shape)
  #print(std_dev)
  randomized_slice = np.empty_like(slice)

  # Randomization in Y
  values = np.arange(slice.shape[0]).astype(np.int32)
  for x in range(slice.shape[1]):
    #pairs = np.array(list(map(tuplify, values, range(len(values)))), dtype=np.int32)
    pairs = shake(values, np.arange(len(values)), std_dev).astype(np.int32)
    pairs = pairs[pairs[:, 0].argsort()]
    randomized_slice[values, x] = slice[pairs[:, 1], x]

  # Randomization in X
  values = np.arange(slice.shape[1]).astype(np.int32)
  for y in range(slice.shape[0]):
    #pairs = np.array(list(map(tuplify, values, range(len(values)))), dtype=np.int32)
     pairs = shake(values, np.arange(len(values)), std_dev).astype(np.int32)
     pairs = pairs[pairs[:, 0].argsort()]
     randomized_slice[y, values] = randomized_slice[y, pairs[:, 1]]

  return randomized_slice

In [ ]:
def normalize(x):
    x_min, x_max = x.min(), x.max()
    return (255.0 * (x - x_min) / (x_max - x_min)).astype(np.uint8)

In [ ]:
def generate_similar(X, std_dev=2.0):
    similar_X = np.zeros_like(X, dtype=np.float32)
    for z in tqdm(range(X.shape[0]), desc=f"std_dev={std_dev}"):
        original_slice = X[z]
        shaked_slice = randomize(slice=original_slice, std_dev=std_dev)

        # Calculate the dense optical flow from slice_z_plus_1 to slice_z
        flow = cv2.calcOpticalFlowFarneback(normalize(original_slice), normalize(shaked_slice), None, **farneback_params)

        # Create a remapping grid from the flow field
        height, width = flow.shape[:2]
        x_coords, y_coords = np.meshgrid(np.arange(width), np.arange(height))

        # The new map tells where each pixel in the output image should come from in the input image
        map_x = (x_coords + flow[..., 0]).astype(np.float32)
        map_y = (y_coords + flow[..., 1]).astype(np.float32)

        # Warp the *original float32 slice* using the map for maximum precision
        projected_slice = cv2.remap(
            src=original_slice,
            map1=map_x,
            map2=map_y,
            #interpolation=cv2.INTER_LINEAR,
            interpolation=cv2.INTER_NEAREST,
            borderMode=cv2.BORDER_REPLICATE # Handle edge pixels
        )

        # Store the result
        similar_X[z, ...] = projected_slice
    return similar_X

In [ ]:
for i in range(N):
    print(i)
    similar_X = generate_similar(X, std_dev=STD_DEV)
    output_filename = f"{i+1:03d}.mrc"
    with mrcfile.new(output_filename, overwrite=True) as mrc:
        mrc.set_data(similar_X)
        mrc.data

In [ ]:
similar_X.shape

In [ ]:
slice_idx = X.shape[0] // 2

fig, axes = plt.subplots(1, 3, figsize=(20, 20))

im0 = axes[0].imshow(X[slice_idx][sliice], cmap='gray', origin='lower')
axes[0].set_title(f'Original Slice z={slice_idx}')
axes[0].grid(False)

im1 = axes[1].imshow(similar_X[slice_idx][sliice], cmap='gray', origin='lower')
axes[1].set_title(f'Projected Slice z={slice_idx}')
axes[1].grid(False)

im2 = axes[2].imshow((X[slice_idx][sliice] - similar_X[slice_idx][sliice] + 128).astype(np.int16), cmap='gray', origin='lower')
axes[2].set_title(f'original[z] - projected[z]')
axes[2].grid(False)

plt.tight_layout()
plt.show()

In [ ]:
!ls -l 001.mrc